# Anime3D Colab — 葉澄汐 img-to-3D 驗證
依 AI_Agent_Colab_Anime3D_Pipeline 執行 N00–N08。預設只檢查與規劃，GPU 後端預設 Pixal3D；Tripo 另需明確付費授權。未核准立繪、缺模型環境、骨架／動畫未通過均不可宣稱完成。

要測試尚未發布的修復，先上傳交付的 `anime3d_colab_source.zip` 至 `/content/`；否則使用 GitHub 指定版本。研究模型請先依官方文件安裝到獨立 Python/CUDA 環境並填入 backend config；此 notebook 不會猜測或混裝模型依賴。


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
REPO = Path('/content/cosmos_chimera_forge')
SOURCE_ARCHIVE = Path('/content/anime3d_colab_source.zip')
REPO_REVISION = 'main'  # 可改成已發布且查核過的 commit SHA
if SOURCE_ARCHIVE.is_file():
    with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
        for item in bundle.infolist():
            target = (REPO / item.filename).resolve()
            if not target.is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe source archive path')
        bundle.extractall(REPO)
    print('Source: uploaded local verification bundle')
elif not REPO.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/WaylenChang/cosmos_chimera_forge.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_REVISION], check=True)
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
assert (REPO / 'scripts/anime3d_preflight.py').is_file(), 'Source revision lacks this notebook support; upload the verification bundle'


In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/project/cosmos_chimera_forge'
ASSET_ID = 'mdl-yeh-chengxi-colab-20261113'
SPEC = str(REPO / 'docs/examples/yeh_chengxi.asset_spec.json')
EXECUTE = False
STAGE = 'gpu'  # pre_gpu (CPU: N00-N01) | gpu (GPU: N02-N04) | post_gpu (CPU: N05-N08) | all
BATCH_QUEUE = ''  # 選填 batch_queue.json 路徑；設定後一次批次跑完再關機
AUTO_SHUTDOWN_GPU = True  # STAGE=='gpu' 完成、佇列為空或閒置逾時時自動關閉 Colab GPU
GPU_IDLE_TIMEOUT_SEC = 180
BACKEND = 'pixal3d'  # pixal3d / anigen / partcrafter / tripo
RIG_BACKEND = 'skintokens'  # Pixal3D mesh → learned skeleton + vertex weights; 'none' for an already rigged input
PIPELINE = 'colab'
BACKEND_CONFIG = ''  # 例如 /content/drive/MyDrive/Anime3D_Agent/backends.json
GENERATOR_EXECUTABLE = ''  # 選填自訂 adapter，可執行檔路徑
EXECUTE_PAID_API = False
INSTALL_BLENDER = True  # 僅 STAGE in ('post_gpu', 'all') 且 EXECUTE=True 時安裝，不在 GPU 階段安裝
BLENDER = shutil.which('blender') or ''
spec = json.loads(Path(SPEC).read_text(encoding='utf8'))
assert spec['input_mode'] == 'image'
print('Subject:', spec.get('subject_name'), spec['subject_id'], 'Source:', spec['source_pack'])


## N00–N02：來源與環境
來源可位於 `01_portraits/` 或 `03_completed/portraits/`；必須有真實 `reviews/<source_pack>.json` 且 status=ok。範例不會替立繪建立核准紀錄。Pixal3D mesh 後接獨立 SkinTokens runtime（官方 demo.py --use_transfer）；請在 BACKEND_CONFIG 同時提供 pixal3d 與 skintokens 設定。參考 docs/anime3d_backends.example.json。原始 mesh 留在 source/，真正 skin/rig 留在 rig/，均以 SHA256 續跑。

SkinTokens 請依 https://github.com/VAST-AI-Research/SkinTokens 在獨立 Python 3.11 環境安裝官方 Torch/CUDA、requirements.txt、flash-attn，再執行 download.py --model。Pixal3D 請依 https://github.com/TencentARC/Pixal3D 安裝 TRELLIS.2 基底、requirements.txt、natten 與 utils3d。兩個模型不可混裝到同一環境。GPU 最低需求依官方與實際 compute capability 檢查；不得因 Colab 分到較小 GPU 就宣稱可推論。

adapter 已提供並有結構／失敗／續跑測試，但尚未通過本角色的真實 GPU 測試。SkinTokens 不保證標準人形骨名；動作仍需依實際骨架映射與視覺驗收，不能把 envelope 骨架或不存在的動畫當正式成果。


In [ ]:
if EXECUTE and INSTALL_BLENDER and not BLENDER and STAGE in ('post_gpu', 'all'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'blender', 'libegl1', 'libgbm1'], check=True)
    BLENDER = shutil.which('blender') or ''
if EXECUTE and BACKEND == 'tripo':
    if not EXECUTE_PAID_API:
        raise RuntimeError('Tripo requires explicit EXECUTE_PAID_API=True')
    key = userdata.get('TRIPO_API_KEY')
    if not key:
        raise RuntimeError('Missing TRIPO_API_KEY in Colab Secrets')
    os.environ['TRIPO_API_KEY'] = key
check_cmd = [sys.executable, str(REPO / 'scripts/anime3d_preflight.py'), '--root', DRIVE_PROJECT,
             '--asset-id', ASSET_ID, '--spec', SPEC, '--backend', BACKEND,
             '--report', f'{DRIVE_PROJECT}/02_3d/colab/{ASSET_ID}/qa/preflight.json']
if BACKEND_CONFIG:
    check_cmd += ['--backend-config', BACKEND_CONFIG]
if RIG_BACKEND != 'none':
    check_cmd += ['--rig-backend', RIG_BACKEND]
if GENERATOR_EXECUTABLE:
    check_cmd += ['--generator-executable', GENERATOR_EXECUTABLE]
if BLENDER:
    check_cmd += ['--blender', BLENDER]
preflight = subprocess.run(check_cmd, check=False)
if EXECUTE and preflight.returncode:
    raise RuntimeError('Preflight blocked execution. Resolve the listed source/runtime checks first.')


## N02–N08：建模、骨架、動作、Blender QA 與匯出
`EXECUTE=False` 僅產生規劃。開源後端需已安裝相容權重和 runtime。`EXECUTE=True` 才啟動推論；Tripo 還需獨立付費旗標。結果保留在 `02_3d/colab/<asset_id>/`，不會自動核准或搬入完成區。


In [ ]:
cmd = [sys.executable, str(REPO / 'scripts/anime3d_universal.py'), '--root', DRIVE_PROJECT,
       '--asset-id', ASSET_ID, '--spec', SPEC, '--backend', BACKEND, '--pipeline', PIPELINE]
_stage = globals().get('STAGE', 'all')
if _stage != 'all':
    cmd += ['--stage', _stage]
if EXECUTE:
    cmd.append('--execute')
if BACKEND == 'tripo' and EXECUTE_PAID_API:
    cmd.append('--allow-paid-tripo')
if BACKEND_CONFIG:
    cmd += ['--backend-config', BACKEND_CONFIG]
if RIG_BACKEND != 'none':
    cmd += ['--rig-backend', RIG_BACKEND]
if GENERATOR_EXECUTABLE:
    cmd += ['--generator-executable', GENERATOR_EXECUTABLE]
if BLENDER and _stage in ('post_gpu', 'all'):
    cmd += ['--blender', BLENDER]
_batch_queue = globals().get('BATCH_QUEUE', '')
_auto_shutdown = globals().get('AUTO_SHUTDOWN_GPU', False)
if _batch_queue:
    _guard_cmd = [sys.executable, str(REPO / 'scripts/anime3d_colab_guard.py'), '--root', DRIVE_PROJECT,
                  '--queue', _batch_queue, '--stage', _stage,
                  '--idle-timeout', str(globals().get('GPU_IDLE_TIMEOUT_SEC', 180))]
    if BACKEND_CONFIG:
        _guard_cmd += ['--backend-config', BACKEND_CONFIG]
    if BLENDER and _stage in ('post_gpu', 'all'):
        _guard_cmd += ['--blender', BLENDER]
    if not EXECUTE:
        _guard_cmd.append('--dry-run')
    if not _auto_shutdown:
        _guard_cmd.append('--no-auto-shutdown')
    subprocess.run(_guard_cmd, check=True)
elif EXECUTE and _stage == 'gpu' and _auto_shutdown:
    sys.path.insert(0, str(REPO / 'scripts'))
    from anime3d_colab_guard import GpuIdleWatchdog, shutdown_colab_runtime
    _wd = GpuIdleWatchdog(idle_timeout_sec=globals().get('GPU_IDLE_TIMEOUT_SEC', 180))
    _wd.set_busy(True)
    _wd.start()
    try:
        subprocess.run(cmd, check=True)  # 失敗必須讓 Colab 顯示錯誤，不能只印 Exit code
    finally:
        _wd.stop()
        shutdown_colab_runtime('gpu_stage_finished')
else:
    subprocess.run(cmd, check=True)  # 失敗必須讓 Colab 顯示錯誤，不能只印 Exit code


In [ ]:
out = Path(DRIVE_PROJECT) / '02_3d/colab' / ASSET_ID
for name in ['typed_state.json', 'qa/typed_metrics.json', 'manifest.json']:
    target = out / name
    if target.is_file():
        result = json.loads(target.read_text(encoding='utf8'))
        print(name, json.dumps({k: result[k] for k in ['status', 'qa_status', 'automated_pass', 'passed', 'missing'] if k in result}, ensure_ascii=False))
print('需檢查模型、三段動畫、四視角與角色臉部／馬尾／背包／右腰珠；自動 QA 不代表人工核准。')
